<a href="https://colab.research.google.com/github/SunshineDalgarno1/Sanger_QCTMB_shasha/blob/main/Shasha_sanger_QCTMB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 📋 Sanger Sequencing Merge, QC & BLAST Pipeline Guide

Welcome to the automated Sanger sequence merging and alignment pipeline! This notebook takes your raw `.seq` or `.ab1` chromatogram files, trims low-quality bases, merges paired reads, trims synthetic primers, and runs a remote NCBI BLAST search to identify viruses, organisms, and potential co-infections.

#### **⚙️ Pipeline Architecture**

```text
       [ Raw .seq or .ab1 Files ]
                   │
                   ▼
       [ Step 1: Quality Control ] ───── ( Trim N's & Phred < 15 )
                   │
                   ▼
       [ Step 2: Primer Trimming ] ───── ( Remove Custom Primers )
                   │
                   ▼
       [ Step 3: Read Merging ] ──────── ( 1-way, 2-way, or N-way )
                   │
                   ├──────────────────────────────────┐
                   ▼                                  ▼
      [ Step 4: Quality Metrics ]        [ Step 5: Remote NCBI BLASTn ]
       ( Mean Q, Q20%, Q30% )          ( Taxonomy & Viral Co-infections )
                   │                                  │
                   └─────────────────┬────────────────┘
                                     ▼
                        [ Step 6: Final Reporting ]
                     ( SUMMARY.csv & merged_sequences )
                                     │
                                     ▼
                      [ Step 7: Visual Verification ]
                        ( In-Notebook Alignments )




```

#### **1. Prepare Your Files**
Gather your sequencing files. The pipeline supports both standard `.seq`/`.fasta` files and raw **`.ab1` chromatogram files**. (Using `.ab1` files is highly recommended, as the script will use the actual Phred quality scores to make smarter merging decisions and generate QC metrics).
*   **Tip:** For the fastest upload, compress all your files into a single `.zip` archive.

#### **2. How to Run the Pipeline**
Run the code cells below in order by clicking the **Play (▶)** button on the left side of each cell.

*   **Cell 1: Setup & Upload**
    Check the **`clear_previous_uploads`** box if you are starting a new batch (this prevents old files from showing up in your dropdown menus). Click play, then select your files to upload.
*   **Cell 1.5: Interactive File Grouper**
    Click play to launch the GUI. Type a Sample ID in the text box, then use the dropdown menus to match your Forward and Reverse reads for that sample. When you are done, click the green **Save to merge.csv** button.
*   **Cell 2: Load Core Functions**
    Click play to load the background sequence processing, QC tracking, and BLAST tools into memory.
*   **Cell 3: Configure & Run**
    Before clicking play, set your parameters on the right side of the cell:
    *   `min_overlap`: Minimum overlapping base pairs required to merge reads (default 40).
    *   `Custom Primers`: Enter the **Name** (which *must* exist inside your filename, e.g., `NP1F`) and the **Sequence** of up to 4 primers to automatically trim them. Leave blank to skip trimming.
    
    Once configured, click play. The pipeline will merge your reads, calculate Q-scores, search the NCBI database, and automatically download a `.zip` file with your results.
*   **Cell 4: View Q-Scores & NCBI Alignments**
    Click play to instantly view your Sample Q-Scores, top BLAST hits, and a visual nucleotide alignment (mimicking the NCBI website) directly inside the notebook.

#### **3. Understanding Your Outputs**
Inside your downloaded `sanger_pipeline_output.zip`, you will find:
*   `SUMMARY.csv`: Your primary report. Includes Sample ID, Merge Type, Quality Metrics (Mean Q, Q20%, Q30%), the top BLAST hit, and the final sequence.
*   `blast_best_hits.tsv`: The top scoring hit for *each unique organism* found per sample. (Ideal for spotting viral co-infections).
*   `blast_results.tsv`: The complete, unfiltered list of all BLAST hits.
*   `merge_summary.tsv`: A highly detailed technical breakdown of lengths, overlaps, mismatches, and merge success.
*   `merged_sequences.fasta`: Your final sequences in standard FASTA format, ready for downstream use.
*   `failed_samples.tsv` *(if applicable)*: A log of any samples that failed to merge and why.

In [ ]:
# @title 1. Setup Environment and Upload Files
# @markdown Check this box to clear old files before uploading a new batch:
clear_previous_uploads = True # @param {type:"boolean"}

# Install biopython to handle .ab1 chromatogram files
!pip install -q biopython

import os
import shutil
from google.colab import files

INPUT_DIR = "sanger_input_seqs"
OUTPUT_DIR = "sanger_pipeline_output"

# Clean up old files if checked
if clear_previous_uploads:
    if os.path.exists(INPUT_DIR):
        shutil.rmtree(INPUT_DIR)
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    if os.path.exists(f"{OUTPUT_DIR}.zip"):
        os.remove(f"{OUTPUT_DIR}.zip")
    print("🧹 Cleared old files from the previous run.")

# Create directories
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Upload your .seq, .ab1 files, and/or a .zip containing them:")
uploaded = files.upload()

# Process uploaded files
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        import zipfile
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(INPUT_DIR)
        os.remove(filename)
        print(f"Extracted {filename} into {INPUT_DIR}/")
    elif filename.lower().endswith((".csv", ".seq", ".ab1", ".fasta", ".fa")):
        shutil.move(filename, os.path.join(INPUT_DIR, filename))

# Flatten directory if a zip contained nested folders
for root, dirs, files_in_dir in os.walk(INPUT_DIR):
    for file in files_in_dir:
        if file.lower().endswith((".seq", ".csv", ".ab1", ".fasta", ".fa")) and root != INPUT_DIR:
            shutil.move(os.path.join(root, file), os.path.join(INPUT_DIR, file))

seq_count = len([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.seq', '.ab1', '.fasta', '.fa'))])
has_csv = "merge.csv" in os.listdir(INPUT_DIR)
print(f"Total sequence files ready: {seq_count}")
if has_csv:
    print("merge.csv detected: Yes")
else:
    print("merge.csv detected: No (You can build one using Cell 1.5 below!)")

In [ ]:
# @title 1.5 Interactive File Grouper (No Excel Needed!)
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import csv

INPUT_DIR = "sanger_input_seqs"
os.makedirs(INPUT_DIR, exist_ok=True)

# Fetch the files the user just uploaded
available_files = [""] + sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.seq', '.ab1', '.fasta', '.fa'))])

rows_container = widgets.VBox([])

def create_row():
    sample_name = widgets.Text(placeholder='Type Sample ID...', layout=widgets.Layout(width='180px'))
    r1 = widgets.Dropdown(options=available_files, layout=widgets.Layout(width='200px'))
    r2 = widgets.Dropdown(options=available_files, layout=widgets.Layout(width='200px'))
    r3 = widgets.Dropdown(options=available_files, layout=widgets.Layout(width='200px'))
    r4 = widgets.Dropdown(options=available_files, layout=widgets.Layout(width='200px'))
    return widgets.HBox([sample_name, r1, r2, r3, r4])

def add_rows(num=5):
    current_rows = list(rows_container.children)
    for _ in range(num):
        current_rows.append(create_row())
    rows_container.children = tuple(current_rows)

# Start with 5 empty rows
add_rows(5)

# Button to add more rows if they have lots of samples
add_btn = widgets.Button(description="+ Add 5 More Samples", button_style='info')
def on_add_click(b):
    add_rows(5)
add_btn.on_click(on_add_click)

# Button to save the CSV
save_btn = widgets.Button(description="💾 Save to merge.csv", button_style='success')
out = widgets.Output()

def on_save(b):
    with out:
        clear_output()
        saved_count = 0
        with open(f"{INPUT_DIR}/merge.csv", "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["Sample ID", "Read 1", "Read 2", "Read 3", "Read 4"])
            for row_box in rows_container.children:
                s_id = row_box.children[0].value.strip()
                r1 = row_box.children[1].value
                r2 = row_box.children[2].value
                r3 = row_box.children[3].value
                r4 = row_box.children[4].value
                # Only save if they typed an ID and selected at least one file
                if s_id and (r1 or r2 or r3 or r4):
                    writer.writerow([s_id, r1, r2, r3, r4])
                    saved_count += 1
        print(f"✅ Success! Saved {saved_count} samples. You can now scroll down and run Cell 3.")

save_btn.on_click(on_save)

header = widgets.HBox([
    widgets.HTML("<b>Sample ID</b>", layout=widgets.Layout(width='180px')),
    widgets.HTML("<b>Read 1 (F)</b>", layout=widgets.Layout(width='200px')),
    widgets.HTML("<b>Read 2 (R)</b>", layout=widgets.Layout(width='200px')),
    widgets.HTML("<b>Read 3</b>", layout=widgets.Layout(width='200px')),
    widgets.HTML("<b>Read 4</b>", layout=widgets.Layout(width='200px')),
])

display(widgets.VBox([
    widgets.HTML("<h3>📝 Interactive File Grouper</h3><p>Type your desired Sample ID, then use the dropdowns to select the files you just uploaded.</p>"),
    header,
    rows_container,
    widgets.HBox([add_btn, save_btn]),
    out
]))

In [ ]:
# @title 2. Load Core Functions
import csv
import json
import pathlib
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

HEADER_RE = re.compile(r"^1st_BASE_(\d+)_(?:H(\d)(\d{3})|([A-Z]\d{2})(\d{3})[A-Z](\d))_(NP[12][FR])$")
RID_RE = re.compile(r"RID\s*=\s*([A-Z0-9]+)")
RTOE_RE = re.compile(r"RTOE\s*=\s*(\d+)")

# Empty default dictionary. Populated in Cell 3 via user input.
PRIMER_SEQS = {}

def read_fasta(path: pathlib.Path) -> tuple[str, str]:
    header = None
    seq_parts = []
    with path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line: continue
            if line.startswith(">"): header = line[1:]
            else: seq_parts.append(line.upper())
    if not header: raise ValueError(f"Missing FASTA header in {path}")
    return header, "".join(seq_parts)

def read_sequence_file(path: pathlib.Path) -> tuple[str, str, list[int] | None]:
    suffix = path.suffix.lower()
    if suffix == ".ab1":
        try:
            from Bio import SeqIO
            record = SeqIO.read(path, "abi")
            header = record.name or record.id or path.stem
            sequence = str(record.seq).upper()
            qualities = None
            if "phred_quality" in record.letter_annotations:
                qualities = list(record.letter_annotations["phred_quality"])
            return header, sequence, qualities
        except Exception as e:
            print(f"Warning: Failed to parse chromatogram {path} via BioPython ({e}). Falling back to FASTA reader.", file=sys.stderr)

    header, sequence = read_fasta(path)
    return header, sequence, None

def calc_q_stats(qualities: list | None) -> dict:
    if not qualities: return {"mean_q": "N/A", "q20_pct": "N/A", "q30_pct": "N/A"}
    phreds = [q * 60.0 if isinstance(q, float) and q <= 1.0 else float(q) for q in qualities]
    if not phreds: return {"mean_q": "N/A", "q20_pct": "N/A", "q30_pct": "N/A"}
    mean_q = sum(phreds) / len(phreds)
    q20 = (sum(1 for q in phreds if q >= 20.0) / len(phreds)) * 100.0
    q30 = (sum(1 for q in phreds if q >= 30.0) / len(phreds)) * 100.0
    return {"mean_q": f"{mean_q:.1f}", "q20_pct": f"{q20:.1f}%", "q30_pct": f"{q30:.1f}%"}

def reverse_complement(seq: str) -> str:
    table = str.maketrans("ACGTN", "TGCAN")
    return seq.translate(table)[::-1]

def trim_quality_and_ambiguous(seq: str, qual: list | None = None, min_q: float = 15.0) -> tuple[str, list | None]:
    if not seq: return seq, qual
    phreds = [q * 60.0 if isinstance(q, float) and q <= 1.0 else float(q) for q in qual] if qual else None
    start, end = 0, len(seq)
    while start < end:
        if seq[start] == "N": start += 1
        elif phreds and phreds[start] < min_q: start += 1
        else: break
    while end > start:
        if seq[end - 1] == "N": end -= 1
        elif phreds and phreds[end - 1] < min_q: end -= 1
        else: break
    return seq[start:end], qual[start:end] if qual is not None else None

def trim_ambiguous(seq: str, qual: list | None = None) -> tuple[str, list | None]:
    return trim_quality_and_ambiguous(seq, qual, min_q=15.0)

def remove_targeted_primers(seq: str, fwd_primer_name: str, rev_primer_name: str, qual: list | None = None) -> tuple[str, list | None]:
    if fwd_primer_name not in PRIMER_SEQS or rev_primer_name not in PRIMER_SEQS:
        return seq, qual

    fwd_primer, rev_primer = PRIMER_SEQS[fwd_primer_name], PRIMER_SEQS[rev_primer_name]
    seq_upper = seq.upper()
    best_start_idx = 0

    core_fwd = fwd_primer[-12:]
    idx_fwd = seq_upper.find(core_fwd, 0, 150)
    if idx_fwd != -1: best_start_idx = idx_fwd + len(core_fwd)

    trimmed_seq = seq[best_start_idx:]
    trimmed_upper = trimmed_seq.upper()

    rc_rev = reverse_complement(rev_primer)
    core_rev = rc_rev[:12]

    best_end_idx = len(trimmed_seq)
    idx_rev = trimmed_upper.find(core_rev)
    if idx_rev != -1: best_end_idx = idx_rev

    res_seq = trimmed_seq[:best_end_idx]
    res_qual = qual[best_start_idx : best_start_idx + best_end_idx] if qual is not None else None
    return res_seq, res_qual

def keep_longest_clean_block(seq: str) -> str:
    if "N" not in seq: return seq
    blocks = seq.split("N")
    if not blocks: return ""
    return max(blocks, key=len)

def get_primer_name(filename: str) -> str | None:
    if not PRIMER_SEQS: return None
    for p in PRIMER_SEQS:
        if p.upper() in filename.upper(): return p
    return None

def process_one_read(seq: str, filename: str, qual: list | None = None, min_q: float = 15.0) -> dict:
    t_seq, t_qual = trim_quality_and_ambiguous(seq, qual, min_q=min_q)
    p_name = get_primer_name(filename)
    if p_name and p_name in PRIMER_SEQS:
        p_seq = PRIMER_SEQS[p_name]
        is_fwd = p_name.endswith("F")
        core = p_seq[-12:] if is_fwd else reverse_complement(p_seq)[:12]
        idx = t_seq.upper().find(core, 0, 150)
        if idx != -1:
            if is_fwd:
                cut = idx + len(core)
                t_seq, t_qual = t_seq[cut:], (t_qual[cut:] if t_qual is not None else None)
            else:
                cut = idx
                t_seq, t_qual = t_seq[:cut], (t_qual[:cut] if t_qual is not None else None)

    clean_seq = keep_longest_clean_block(t_seq)
    q_stats = calc_q_stats(t_qual)
    return {
        "merged_sequence": clean_seq, "qualities": t_qual, "overlap": len(clean_seq),
        "identity": 1.0, "informative_bases": len(clean_seq), "mismatches": 0, "q_stats": q_stats,
    }

def score_overlap(a: str, b: str) -> tuple[int, float, int]:
    matches, informative, mismatches = 0, 0, 0
    for base_a, base_b in zip(a, b):
        if "N" in (base_a, base_b): continue
        informative += 1
        if base_a == base_b: matches += 1
        else: mismatches += 1
    if informative == 0: return -10**9, 0.0, 0
    return matches - (2 * mismatches), matches / informative, informative

def choose_base(base_a: str, base_b: str) -> str:
    if base_a == base_b: return base_a
    if base_a == "N": return base_b
    if base_b == "N": return base_a
    return base_a

def merge_pair(seq_a: str, seq_b: str, q_a_in=None, q_b_in=None, min_overlap: int = 40, reverse_second: bool = True) -> dict:
    seq_b_oriented = reverse_complement(seq_b) if reverse_second else seq_b
    best = None
    len_a, len_b = len(seq_a), len(seq_b_oriented)

    q_a = q_a_in if q_a_in is not None else [30.0 for _ in range(len_a)]
    q_b = q_b_in if q_b_in is not None else [30.0 for _ in range(len_b)]
    q_b_oriented = q_b[::-1] if reverse_second else q_b

    for shift in range(-len_b + min_overlap, len_a - min_overlap + 1):
        start_a, start_b = max(0, shift), max(0, -shift)
        overlap = min(len_a - start_a, len_b - start_b)
        if overlap < min_overlap: continue
        seg_a, seg_b = seq_a[start_a:start_a + overlap], seq_b_oriented[start_b:start_b + overlap]
        score, identity, informative = score_overlap(seg_a, seg_b)
        candidate = (score, identity, informative, overlap, shift)
        if best is None or candidate > best: best = candidate

    if best is None: raise ValueError("No acceptable overlap found between reads")

    _, identity, informative, overlap, shift = best
    start, end = min(0, shift), max(len_a, shift + len_b)
    merged, merged_q, mismatches = [], [], 0

    for pos in range(start, end):
        base_a, base_b = "N", "N"
        idx_a, idx_b = pos, pos - shift

        val_q_a, val_q_b = 0.0, 0.0
        if 0 <= idx_a < len_a: base_a, val_q_a = seq_a[idx_a], q_a[idx_a]
        if 0 <= idx_b < len_b: base_b, val_q_b = seq_b_oriented[idx_b], q_b_oriented[idx_b]

        if base_a != "N" and base_b != "N" and base_a != base_b: mismatches += 1

        if base_a == base_b: chosen, chosen_q = base_a, max(val_q_a, val_q_b)
        elif base_a == "N": chosen, chosen_q = base_b, val_q_b
        elif base_b == "N": chosen, chosen_q = base_a, val_q_a
        else:
            if val_q_a >= val_q_b: chosen, chosen_q = base_a, val_q_a
            else: chosen, chosen_q = base_b, val_q_b

        merged.append(chosen)
        merged_q.append(chosen_q)

    seq_str = "".join(merged)
    merged_sequence = seq_str.strip("N")
    start_strip = seq_str.find(merged_sequence)
    end_strip = start_strip + len(merged_sequence)

    return {
        "merged_sequence": merged_sequence, "qualities": merged_q[start_strip:end_strip],
        "overlap": overlap, "identity": identity, "informative_bases": informative,
        "shift": shift, "mismatches": mismatches, "reverse_second": reverse_second,
    }

def submit_blast(query_fasta: str, database: str = "nt") -> tuple[str, int | None]:
    params = urllib.parse.urlencode({"CMD": "Put", "PROGRAM": "blastn", "DATABASE": database, "QUERY": query_fasta}).encode()
    request = urllib.request.Request("https://blast.ncbi.nlm.nih.gov/Blast.cgi", data=params, method="POST")
    request.add_header("User-Agent", "Mozilla/5.0")
    with urllib.request.urlopen(request, timeout=60) as response: text = response.read().decode()
    rid_match, rtoe_match = RID_RE.search(text), RTOE_RE.search(text)
    if not rid_match: raise RuntimeError("NCBI BLAST did not return an RID")
    return rid_match.group(1), int(rtoe_match.group(1)) if rtoe_match else None

def fetch_blast_results(rid: str, poll_seconds: int = 20, max_wait: int = 600) -> str:
    waited = 0
    while waited <= max_wait:
        status_query = urllib.parse.urlencode({"CMD": "Get", "RID": rid, "FORMAT_OBJECT": "SearchInfo"})
        request = urllib.request.Request(f"https://blast.ncbi.nlm.nih.gov/Blast.cgi?{status_query}")
        request.add_header("User-Agent", "Mozilla/5.0")
        with urllib.request.urlopen(request, timeout=60) as response: text = response.read().decode()
        if "Status=WAITING" in text:
            time.sleep(poll_seconds)
            waited += poll_seconds
            continue
        if "Status=FAILED" in text: raise RuntimeError(f"NCBI BLAST RID {rid} failed")
        if "Status=UNKNOWN" in text: raise RuntimeError(f"NCBI BLAST RID {rid} expired or is unknown")
        if "Status=READY" in text and "ThereAreHits=yes" in text:
            result_query = urllib.parse.urlencode({"RESULTS_FILE": "on", "CMD": "Get", "RID": rid, "FORMAT_TYPE": "CSV", "DESCRIPTIONS": "5", "FORMAT_OBJECT": "Alignment", "ALIGNMENT_VIEW": "Tabular"})
            result_request = urllib.request.Request(f"https://blast.ncbi.nlm.nih.gov/Blast.cgi?{result_query}")
            result_request.add_header("User-Agent", "Mozilla/5.0")
            with urllib.request.urlopen(result_request, timeout=60) as response: return response.read().decode()
        if "Status=READY" in text and "ThereAreHits=no" in text: return ""
        time.sleep(poll_seconds)
        waited += poll_seconds
    raise TimeoutError(f"Timed out waiting for BLAST RID {rid}")

def parse_blast_tabular(text: str) -> list[list[str]]:
    return [row for row in csv.reader(text.splitlines()) if row]

def fetch_nuccore_summaries(accessions: list[str]) -> dict[str, dict[str, str]]:
    if not accessions: return {}
    summary_query = urllib.parse.urlencode({"db": "nuccore", "id": ",".join(accessions), "retmode": "json"})
    request = urllib.request.Request(f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?{summary_query}")
    request.add_header("User-Agent", "Mozilla/5.0")
    with urllib.request.urlopen(request, timeout=60) as response: payload = json.loads(response.read().decode())
    by_accession = {}
    result = payload.get("result", {})
    for uid in result.get("uids", []):
        item = result.get(uid, {})
        accession = item.get("accessionversion") or item.get("caption")
        if accession: by_accession[accession] = item
    return by_accession

def extract_gene_name(title: str, organism: str) -> str:
    before_comma = title.split(",", 1)[0].strip()
    tokens = before_comma.split()
    for i in range(len(tokens) - 1, -1, -1):
        if tokens[i].lower() == "gene":
            start = max(0, i - 7)
            candidate = " ".join(tokens[start : i + 1])
            candidate = re.sub(r"^(isolate|strain|clone|segment)\s+\S+\s+", "", candidate, flags=re.IGNORECASE)
            if organism and candidate.lower().startswith(organism.lower()): candidate = candidate[len(organism) :].strip()
            return candidate.strip() or before_comma
    return before_comma

def load_merge_csv(path: pathlib.Path) -> list[dict]:
    records = []
    with path.open() as handle:
        reader = csv.reader(handle)
        header = next(reader, None)
        if not header: return []
        sample_col_idx = -1
        read_col_indexes = []
        for idx, col in enumerate(header):
            col_upper = col.strip().upper()
            if col_upper in ("SAMPLE ID", "SAMPLE_ID", "SAMPLE", "ID"): sample_col_idx = idx
            elif col_upper.startswith("READ"): read_col_indexes.append(idx)
        if sample_col_idx == -1:
            sample_col_idx = 0
            read_col_indexes = list(range(1, len(header)))
        for row in reader:
            if not row: continue
            sample_id = row[sample_col_idx].strip()
            files = [row[idx].strip() for idx in read_col_indexes if idx < len(row) and row[idx].strip()]
            if sample_id and files: records.append({"sample_id": sample_id, "files": files})
    return records

def merge_two_reads(seq1: str, name1: str, seq2: str, name2: str, min_overlap: int, q1_in: list[float] | None = None, q2_in: list[float] | None = None) -> dict:
    p1, p2 = get_primer_name(name1), get_primer_name(name2)
    if p1 and p2:
        is_f1, is_f2 = p1.endswith("F"), p2.endswith("F")
        if is_f1 and not is_f2:
            fwd_seq, fwd_q = remove_targeted_primers(seq1, p1, p2, q1_in)
            rev_seq, rev_q = remove_targeted_primers(seq2, p2, p1, q2_in)
            return merge_pair(fwd_seq, rev_seq, q_a_in=fwd_q, q_b_in=rev_q, reverse_second=True, min_overlap=min_overlap)
        elif is_f2 and not is_f1:
            fwd_seq, fwd_q = remove_targeted_primers(seq2, p2, p1, q2_in)
            rev_seq, rev_q = remove_targeted_primers(seq1, p1, p2, q1_in)
            return merge_pair(fwd_seq, rev_seq, q_a_in=fwd_q, q_b_in=rev_q, reverse_second=True, min_overlap=min_overlap)

    best_res, best_score = None, -10**9
    for rev in (False, True):
        try:
            res = merge_pair(seq1, seq2, q_a_in=q1_in, q_b_in=q2_in, reverse_second=rev, min_overlap=min_overlap)
            score = res["overlap"] - 3 * res["mismatches"]
            if score > best_score: best_score, best_res = score, res
        except ValueError: pass
    if best_res is None: raise ValueError(f"No acceptable overlap found between {name1} and {name2}")
    return best_res

In [ ]:
# @title 3. Run Pipeline and Download Results
# @markdown Configure your settings:
min_overlap = 40 # @param {type:"integer"}
skip_blast = False # @param {type:"boolean"}
download_results_when_done = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### **Add Custom Primers**
# @markdown *(The **Name** must be present inside your .seq or .ab1 filename for the script to use it)*

# @markdown **Primer 1**
primer_1_name = "AN88" # @param {type:"string"}
primer_1_seq = "TACTGGACCACCTGGNGGNAYRWACAT" # @param {type:"string"}

# @markdown **Primer 2**
primer_2_name = "" # @param {type:"string"}
primer_2_seq = "" # @param {type:"string"}

# @markdown **Primer 3**
primer_3_name = "" # @param {type:"string"}
primer_3_seq = "" # @param {type:"string"}

# @markdown **Primer 4**
primer_4_name = "" # @param {type:"string"}
primer_4_seq = "" # @param {type:"string"}

input_dir = pathlib.Path(INPUT_DIR)
output_dir = pathlib.Path(OUTPUT_DIR)

# Update the global PRIMER_SEQS with any user input
PRIMER_SEQS.clear()
for p_name, p_seq in [(primer_1_name, primer_1_seq), (primer_2_name, primer_2_seq), (primer_3_name, primer_3_seq), (primer_4_name, primer_4_seq)]:
    if p_name.strip() and p_seq.strip():
        PRIMER_SEQS[p_name.strip()] = p_seq.strip().upper()

csv_path = input_dir / "merge.csv"
if not csv_path.exists():
    print("Error: merge.csv not found in the uploaded files. Please upload it in Cell 1.")
else:
    jobs, unmatched, merged_records = [], [], []

    print("Reading merge.csv and processing sequences...")
    csv_records = load_merge_csv(csv_path)
    for r in csv_records:
        valid_files = []
        for f in r["files"]:
            p_in = input_dir / f
            p_in_ab1 = p_in.with_suffix(".ab1")

            if p_in.suffix.lower() == ".seq" and p_in_ab1.is_file(): valid_files.append(p_in_ab1)
            elif p_in.is_file(): valid_files.append(p_in)
            else: print(f"Warning: file {f} not found for sample {r['sample_id']}", file=sys.stderr)

        # Supports 1 or more files now
        if len(valid_files) >= 1:
            prefix_match = re.search(r"^[A-Z]\d{2}", r["sample_id"])
            jobs.append({
                "sample_id": r["sample_id"],
                "files": valid_files,
                "prefix": prefix_match.group(0) if prefix_match else "B26"
            })

    merged_fasta_path = output_dir / "merged_sequences.fasta"
    summary_path = output_dir / "merge_summary.tsv"

    with merged_fasta_path.open("w") as fasta_handle, summary_path.open("w", newline="") as summary_handle:
        writer = csv.writer(summary_handle, delimiter="\t")
        writer.writerow(["sample_suffix", "read_type", "h1_forward_file", "h1_reverse_file", "h1_length", "h1_overlap", "h1_identity", "h1_informative_bases", "h1_mismatches", "h2_forward_file", "h2_reverse_file", "h2_length", "h2_overlap", "h2_identity", "h2_informative_bases", "h2_mismatches", "final_length", "final_overlap", "final_identity", "final_informative_bases", "final_mismatches", "mean_q_score", "q20_percent", "q30_percent"])

        for job in jobs:
            sample_id = job["sample_id"]
            job_files = job["files"]

            sequences, qualities_list = [], []
            for f in job_files:
                _, seq, qual = read_sequence_file(f)
                t_seq, t_qual = trim_quality_and_ambiguous(seq, qual, min_q=15.0)
                sequences.append(t_seq)
                qualities_list.append(t_qual)

            try:
                # 1-WAY MERGE
                if len(job_files) == 1:
                    res = process_one_read(sequences[0], job_files[0].name, qualities_list[0], min_q=15.0)
                    clean_final_seq, q_stats = res["merged_sequence"], res["q_stats"]

                    fasta_handle.write(f">{sample_id}\n{clean_final_seq}\n")
                    merged_records.append((sample_id, clean_final_seq, q_stats, "1-way"))

                    writer.writerow([sample_id, "1-way", job_files[0].name, "N/A", len(clean_final_seq), "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", len(clean_final_seq), "N/A", "1.0000", len(clean_final_seq), 0, q_stats["mean_q"], q_stats["q20_pct"], q_stats["q30_pct"]])
                    continue

                # 4-WAY MERGE
                if len(job_files) == 4:
                    mapped_seqs, mapped_quals, mapped_paths = {}, {}, {}
                    for f, seq, qual in zip(job_files, sequences, qualities_list):
                        p = get_primer_name(f.name)
                        if p:
                            mapped_seqs[p], mapped_quals[p], mapped_paths[p] = seq, qual, f.name

                    if len(mapped_seqs) == 4:
                        h1_seq_fwd, h1_q_fwd = remove_targeted_primers(mapped_seqs["NP1F"], "NP1F", "NP2R", mapped_quals.get("NP1F"))
                        h1_seq_rev, h1_q_rev = remove_targeted_primers(mapped_seqs["NP2R"], "NP2R", "NP1F", mapped_quals.get("NP2R"))
                        h2_seq_fwd, h2_q_fwd = remove_targeted_primers(mapped_seqs["NP2F"], "NP2F", "NP1R", mapped_quals.get("NP2F"))
                        h2_seq_rev, h2_q_rev = remove_targeted_primers(mapped_seqs["NP1R"], "NP1R", "NP2F", mapped_quals.get("NP1R"))

                        h1_merged = merge_pair(h1_seq_fwd, h1_seq_rev, q_a_in=h1_q_fwd, q_b_in=h1_q_rev, reverse_second=True, min_overlap=min_overlap)
                        h2_merged = merge_pair(h2_seq_fwd, h2_seq_rev, q_a_in=h2_q_fwd, q_b_in=h2_q_rev, reverse_second=True, min_overlap=min_overlap)

                        final_merged = merge_pair(h1_merged["merged_sequence"], h2_merged["merged_sequence"], q_a_in=h1_merged["qualities"], q_b_in=h2_merged["qualities"], reverse_second=False, min_overlap=min_overlap)
                        clean_final_seq = keep_longest_clean_block(final_merged["merged_sequence"])
                        q_stats = calc_q_stats(final_merged.get("qualities"))

                        fasta_handle.write(f">{sample_id}\n{clean_final_seq}\n")
                        merged_records.append((sample_id, clean_final_seq, q_stats, "4-way"))

                        writer.writerow([sample_id, "4-way", mapped_paths["NP1F"], mapped_paths["NP2R"], len(h1_merged["merged_sequence"]), h1_merged["overlap"], f"{h1_merged['identity']:.4f}", h1_merged["informative_bases"], h1_merged["mismatches"], mapped_paths["NP2F"], mapped_paths["NP1R"], len(h2_merged["merged_sequence"]), h2_merged["overlap"], f"{h2_merged['identity']:.4f}", h2_merged["informative_bases"], h2_merged["mismatches"], len(clean_final_seq), final_merged["overlap"], f"{final_merged['identity']:.4f}", final_merged["informative_bases"], final_merged["mismatches"], q_stats["mean_q"], q_stats["q20_pct"], q_stats["q30_pct"]])
                        continue

                # 2-WAY MERGE
                if len(job_files) == 2:
                    res = merge_two_reads(sequences[0], job_files[0].name, sequences[1], job_files[1].name, min_overlap, q1_in=qualities_list[0], q2_in=qualities_list[1])
                    clean_final_seq = keep_longest_clean_block(res["merged_sequence"])
                    q_stats = calc_q_stats(res.get("qualities"))

                    fasta_handle.write(f">{sample_id}\n{clean_final_seq}\n")
                    merged_records.append((sample_id, clean_final_seq, q_stats, "2-way"))
                    writer.writerow([sample_id, "2-way", job_files[0].name, job_files[1].name, len(clean_final_seq), res["overlap"], f"{res['identity']:.4f}", res["informative_bases"], res["mismatches"], "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", len(clean_final_seq), res["overlap"], f"{res['identity']:.4f}", res["informative_bases"], res["mismatches"], q_stats["mean_q"], q_stats["q20_pct"], q_stats["q30_pct"]])

                # N-WAY MERGE
                else:
                    current_seq = sequences[0]
                    current_q = qualities_list[0] if qualities_list[0] is not None else [30.0 for _ in range(len(current_seq))]
                    last_res = None
                    for next_name, next_seq, next_q in zip(job_files[1:], sequences[1:], qualities_list[1:]):
                        best_res, best_score = None, -10**9
                        for rev in (False, True):
                            try:
                                res = merge_pair(current_seq, next_seq, q_a_in=current_q, q_b_in=next_q, min_overlap=min_overlap, reverse_second=rev)
                                score = res["overlap"] - 3 * res["mismatches"]
                                if score > best_score: best_score, best_res = score, res
                            except ValueError: pass
                        if not best_res: raise ValueError(f"Could not merge read {next_name.name} into consensus")
                        current_seq, current_q, last_res = best_res["merged_sequence"], best_res["qualities"], best_res

                    clean_final_seq = keep_longest_clean_block(current_seq)
                    q_stats = calc_q_stats(last_res.get("qualities") if last_res else current_q)

                    fasta_handle.write(f">{sample_id}\n{clean_final_seq}\n")
                    merged_records.append((sample_id, clean_final_seq, q_stats, f"{len(job_files)}-way"))
                    writer.writerow([sample_id, f"{len(job_files)}-way", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", len(clean_final_seq), last_res["overlap"] if last_res else "N/A", f"{last_res['identity']:.4f}" if last_res else "N/A", last_res["informative_bases"] if last_res else "N/A", last_res["mismatches"] if last_res else "N/A", q_stats["mean_q"], q_stats["q20_pct"], q_stats["q30_pct"]])

            except ValueError as e:
                print(f"Error merging sample {sample_id}: {e}", file=sys.stderr)
                unmatched.append((sample_id, str(e)))

    best_by_query_org = {}

    if not skip_blast and merged_records:
        print("\nRunning remote NCBI BLAST...")
        blast_path = output_dir / "blast_results.tsv"
        best_hits_path = output_dir / "blast_best_hits.tsv"
        blast_errors_path = output_dir / "blast_errors.tsv"
        blast_errors = []
        try:
            query_fasta = "".join(f">{sample_code}\n{sequence}\n" for sample_code, sequence, _, _ in merged_records)
            rid, rtoe = submit_blast(query_fasta)
            print(f"Submitted BLAST RID {rid}", flush=True)
            result_text = fetch_blast_results(rid, poll_seconds=max(10, rtoe or 20))
            blast_rows = parse_blast_tabular(result_text)

            accessions = list({row[1] for row in blast_rows if len(row) >= 2})
            summary_by_accession = fetch_nuccore_summaries(accessions) if accessions else {}

            headers = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore", "gene", "organism", "subject_title"]

            with blast_path.open("w", newline="") as blast_handle:
                writer = csv.writer(blast_handle, delimiter="\t")
                writer.writerow(headers)
                for row in blast_rows:
                    if len(row) < 12: continue
                    accession = row[1]
                    summary = summary_by_accession.get(accession, {})
                    title = summary.get("title", "")
                    organism = summary.get("organism", "Unknown Organism")
                    gene = extract_gene_name(title, organism) if title else ""
                    writer.writerow(row + [gene, organism, title])

                    # Track best hit per unique (Sample_ID, Organism)
                    query = row[0]
                    key = (query, organism)
                    if key not in best_by_query_org:
                        best_by_query_org[key] = row + [gene, organism, title]

            # Write the best hits grouped by organism
            with best_hits_path.open("w", newline="") as best_handle:
                writer = csv.writer(best_handle, delimiter="\t")
                writer.writerow(headers)

                # Sort output by Sample ID first, then by highest bitscore
                def sort_best_hits(item):
                    query_id = item[0]
                    try:
                        bitscore = float(item[11])
                    except ValueError:
                        bitscore = 0.0
                    return (query_id, -bitscore)

                sorted_hits = sorted(best_by_query_org.values(), key=sort_best_hits)
                for hit in sorted_hits:
                    writer.writerow(hit)

        except Exception as exc:
            print(f"BLAST Error: {exc}")
            blast_errors.append(["ALL", type(exc).__name__, str(exc)])

        if blast_errors:
            with blast_errors_path.open("w", newline="") as error_handle:
                csv.writer(error_handle, delimiter="\t").writerows([["sample_code", "error_type", "message"]] + blast_errors)

    summary_csv_path = output_dir / "SUMMARY.csv"
    with summary_csv_path.open("w", newline="") as summary_csv_handle:
        writer = csv.writer(summary_csv_handle)
        writer.writerow(["Sample ID", "Read Type", "Mean Q-score", "Q20 (%)", "Q30 (%)", "Blast result (Best hit)", "Sequence (fasta format)"])
        for record in merged_records:
            sample_id, seq, q_stats, read_type = record[0], record[1], record[2], record[3]
            best_hit = "N/A"
            if not skip_blast:
                sample_hits = [hit for key, hit in best_by_query_org.items() if key[0] == sample_id]
                if sample_hits:
                    top_hit_for_sample = max(sample_hits, key=lambda x: float(x[11]) if x[11].replace('.','',1).isdigit() else 0.0)
                    best_hit = top_hit_for_sample[14]
            writer.writerow([sample_id, read_type, q_stats["mean_q"], q_stats["q20_pct"], q_stats["q30_pct"], best_hit, f">{sample_id}\n{seq}"])

    if unmatched:
        with (output_dir / "unmatched_inputs.tsv").open("w", newline="") as handle:
            csv.writer(handle, delimiter="\t").writerows([["item", "reason"]] + unmatched)

    print("\n--- Pipeline Complete ---")
    if download_results_when_done:
        print("Zipping and downloading results...")
        shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
        from google.colab import files as colab_files
        colab_files.download(f"{OUTPUT_DIR}.zip")

In [ ]:
# @title 4. View Q-Scores and NCBI Alignments
import pandas as pd
import urllib.request
from io import StringIO
import time
import re
from IPython.display import display, HTML

# Safely import Biopython
try:
    from Bio import SeqIO, Align
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "biopython"])
    from Bio import SeqIO, Align

OUTPUT_DIR = "sanger_pipeline_output"
summary_csv = f"{OUTPUT_DIR}/SUMMARY.csv"
blast_tsv = f"{OUTPUT_DIR}/blast_best_hits.tsv"
fasta_path = f"{OUTPUT_DIR}/merged_sequences.fasta"

try:
    summary_df = pd.read_csv(summary_csv)
    blast_df = pd.read_csv(blast_tsv, sep="\t")

    # FORCE everything to string to prevent numerical ID mismatches
    summary_df['Sample ID'] = summary_df['Sample ID'].astype(str)
    if 'qseqid' in blast_df.columns:
        blast_df['qseqid'] = blast_df['qseqid'].astype(str)

    fasta_dict = {str(rec.id): str(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")}
except Exception as e:
    print("❌ Could not load output files. Please ensure Cell 3 completed successfully.")
    raise e

print("Fetching reference sequences from NCBI... (This may take a moment)\n")

for idx, row in summary_df.iterrows():
    sample_id = row['Sample ID']
    print("="*85)
    print(f"🧬 Sample Name : {sample_id}")

    mean_q = row.get('Mean Q-score', 'N/A')
    q20 = row.get('Q20 (%)', 'N/A')
    q30 = row.get('Q30 (%)', 'N/A')

    print(f"📊 Q-Scores    : Mean = {mean_q} | Q20 = {q20} | Q30 = {q30}")

    if 'qseqid' not in blast_df.columns:
        print("❌ BLAST results file is empty or corrupted.")
        continue

    hit_rows = blast_df[blast_df['qseqid'] == sample_id]
    if hit_rows.empty:
        print("❌ No BLAST hit found for this sample.")
        continue

    # Get highest bitscore hit
    top_hit = hit_rows.sort_values(by='bitscore', ascending=False).iloc[0]
    accession = top_hit['sseqid']
    sstart = int(top_hit['sstart'])
    send = int(top_hit['send'])
    organism = top_hit['organism']
    pident = top_hit['pident']

    print(f"🦠 Best Hit    : {organism} ({accession})")
    print(f"💯 Identity    : {pident}%")
    print(f"🔗 NCBI Link   : https://www.ncbi.nlm.nih.gov/nuccore/{accession}")

    query_seq = fasta_dict.get(sample_id, "")
    if not query_seq:
        print(f"⚠️ Could not find sequence for {sample_id} to build alignment.")
        continue

    # Fetch sequence from NCBI (only the part that matched)
    strand = 1 if sstart < send else 2
    start, stop = min(sstart, send), max(sstart, send)
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={accession}&rettype=fasta&retmode=text&seq_start={start}&seq_stop={stop}&strand={strand}"

    ref_seq = None
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=10) as response:
                fasta_data = response.read().decode('utf-8')
            record = SeqIO.read(StringIO(fasta_data), "fasta")
            ref_seq = str(record.seq)
            break
        except Exception:
            time.sleep(1.5) # Wait to avoid NCBI rate limits

    if not ref_seq:
        print("\n⚠️ Could not fetch reference sequence from NCBI to build alignment visual.")
        continue

    print("\n--- Alignment ( | = Match ) ---\n")

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.match_score = 1
    aligner.mismatch_score = -2
    aligner.open_gap_score = -2
    aligner.extend_gap_score = -0.5

    alignments = aligner.align(ref_seq, query_seq)
    best_alignment = alignments[0]

    # Extract strings from Biopython's default formatter
    raw_str = str(best_alignment).rstrip().split('\n')

    if len(raw_str) >= 3:
        # Regex to pull out just the sequence strings, ignoring coordinate numbers
        t_match = re.search(r'target\s+\d+\s+([A-Za-z-]+)\s+\d+', raw_str[0])
        q_match = re.search(r'query\s+\d+\s+([A-Za-z-]+)\s+\d+', raw_str[2])

        if t_match and q_match:
            t_seq = t_match.group(1)
            q_seq = q_match.group(1)

            # Find the match indicator line based on the start position
            start_idx = raw_str[0].find(t_seq)
            m_seq = raw_str[1][start_idx : start_idx + len(t_seq)]

            # Wrap the alignment exactly like NCBI BLAST (60 bases per row)
            wrap = 60
            for i in range(0, len(t_seq), wrap):
                chunk_t = t_seq[i:i+wrap]
                chunk_m = m_seq[i:i+wrap]
                chunk_q = q_seq[i:i+wrap]

                # Print with nice padding
                print(f"Ref   {i+1:04d} {chunk_t}")
                print(f"           {chunk_m}")
                print(f"Query {i+1:04d} {chunk_q}\n")
        else:
            print(str(best_alignment))
    else:
        print(str(best_alignment))